In [1]:
import pandas as pd
import numpy as np
import xarray as xr

from scripts.build_transport_demand import (
    build_nodal_transport_data,
    build_transport_demand,
    transport_degree_factor,
    bev_availability_profile,
    bev_dsm_profile,
)

# Dummy pop_layout DataFrame
# pop_layout = pd.DataFrame({
#     "ct": ["BE", "BE", "BE"],
#     "fraction": [0.6, 1.0, 0.4]
# }, index=["bus1", "bus2", "bus3"])


pop_layout = pd.read_csv("dummy_pop_layout_base_s_5.csv", index_col=0)
# pop_layout = pd.DataFrame([
#     {"name": "BE0 0", "total": 1481.365696082056, "urban": 1518.7004503118183, "rural": 10.941661286650039, "ct": "BE", "fraction": 0.1422218074818682},
#     {"name": "BE0 1", "total": 4649.126947856696, "urban": 4761.693149742594, "rural": 10.662264329847897, "ct": "BE", "fraction": 0.4463497700031895},
#     {"name": "BE0 2", "total": 1107.7329541679119, "urban": 1189.5941988980433, "rural": 1.9207033772297681, "ct": "BE", "fraction": 0.10635036531874921},
#     {"name": "BE0 3", "total": 488.53973036203575, "urban": 439.351533000744, "rural": 53.099663179197414, "ct": "BE", "fraction": 0.046903343085747104},
#     {"name": "BE0 4", "total": 2689.1175963135274, "urban": 2911.348763502409, "rural": 23.09985733255275, "ct": "BE", "fraction": 0.258174714110446},   
# ])

# Dummy transport_data CSV
# transport_data = pd.DataFrame({
#     ("DE", 2030): {"number cars": 1000, "average fuel efficiency": 0.15},
#     ("FR", 2030): {"number cars": 800, "average fuel efficiency": 0.0}
# }).T
# transport_data.index.names = ["ct", "year"]
# transport_data.to_csv("dummy_transport_data.csv")

# Dummy traffic data
with open("dummy_traffic.csv", "w") as f:
    f.write("ignore,this\n")
    f.write("ignore,this\n")
    f.write("count\n")
    for i in range(168):
        f.write(f"{np.random.randint(100,200)}\n")

# Dummy temperature data (xarray DataArray saved as netCDF)
# temperature = xr.DataArray(np.random.uniform(10, 25, size=168), dims=["time"])
# temperature.to_netcdf("dummy_temperature.nc")

# Dummy pop_weighted_energy_totals
pop_weighted_energy_totals = pd.DataFrame({
    "total road": [10, 20, 30],
    "total rail": [2, 3, 4],
    "electricity rail": [1, 1, 1]
}, index=["bus1", "bus2", "bus3"])

# Dummy options
options = {
    "transport_heating_deadband_lower": 15,
    "transport_heating_deadband_upper": 20,
    "ICE_lower_degree_factor": 0.5,
    "ICE_upper_degree_factor": 1.6,
    "bev_avail_max": 0.9,
    "bev_avail_mean": 0.5,
    "bev_dsm_restriction_time": 18,
    "bev_dsm_restriction_value": 0.3,
}

# Dummy snapshots and nodes
snapshots = pd.date_range("2013-03-01", periods=168, freq="H", tz="UTC")
nodes = pop_layout.index

# Test build_nodal_transport_data
energy_totals_year = 2013
nodal_transport_data = build_nodal_transport_data("dummy_transport_data.csv", pop_layout, year=energy_totals_year)
print("Nodal Transport Data:\n", nodal_transport_data)

# Test transport_degree_factor
# temperature_pd = pd.Series(np.random.uniform(10, 25, size=168), index=snapshots)
temperature_pd = xr.open_dataarray("dummy_temperature.nc").to_pandas()
dd = transport_degree_factor(
    temperature_pd,
    options["transport_heating_deadband_lower"],
    options["transport_heating_deadband_upper"],
    options["ICE_lower_degree_factor"],
    options["ICE_upper_degree_factor"],
)
print("Degree Factor:\n", dd)

/var/folders/vg/7b9brk_5419g9__hdm5j45vc0000gn/T/ipykernel_39162/1938660071.py:69: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



Nodal Transport Data:
         number cars  average fuel efficiency
name                                        
BE0 0  7.812915e+05                  0.06876
BE0 1  2.452010e+06                  0.06876
BE0 2  5.842328e+05                  0.06876
BE0 3  2.576622e+05                  0.06876
BE0 4  1.418276e+06                  0.06876
Degree Factor:
 name                    BE0 0     BE0 1     BE0 2     BE0 3     BE0 4
time                                                                 
2013-03-01 00:00:00  0.069706  0.065367  0.067166  0.075051  0.068335
2013-03-01 01:00:00  0.070874  0.066071  0.068154  0.076378  0.069376
2013-03-01 02:00:00  0.071332  0.066209  0.068853  0.076163  0.069480
2013-03-01 03:00:00  0.071438  0.066905  0.069042  0.076802  0.070185
2013-03-01 04:00:00  0.071828  0.067327  0.069095  0.077573  0.071149
...                       ...       ...       ...       ...       ...
2013-03-07 19:00:00  0.022117  0.023135  0.021502  0.028879  0.023537
2013-03-07 20:00

In [2]:
# Test bev_availability_profile
avail_profile = bev_availability_profile("dummy_traffic.csv", snapshots, nodes, options)
print("BEV Availability Profile:\n", avail_profile)

# Test bev_dsm_profile
dsm_profile = bev_dsm_profile(snapshots, nodes, options)
print("BEV DSM Profile:\n", dsm_profile)

pop_weighted_energy_totals = pd.read_csv("dummy_pop_weighted_energy_totals.csv", index_col=0)
nyears = len(snapshots) / 8760

# Test build_transport_demand (hier musst du ggf. die Funktion anpassen, da sie pop_weighted_energy_totals und nyears erwartet)
# Für einen einfachen Test kannst du die Funktion so aufrufen:
transport_demand = build_transport_demand("dummy_traffic.csv", "dummy_temperature.nc", snapshots, nodes, nodal_transport_data, options, pop_weighted_energy_totals, nyears)
print("Transport Demand:\n", transport_demand)

BEV Availability Profile:
 name                    BE0 0     BE0 1     BE0 2     BE0 3     BE0 4
2013-03-01 00:00:00  0.700717  0.700717  0.700717  0.700717  0.700717
2013-03-01 01:00:00  0.584468  0.584468  0.584468  0.584468  0.584468
2013-03-01 02:00:00  0.185901  0.185901  0.185901  0.185901  0.185901
2013-03-01 03:00:00  0.169294  0.169294  0.169294  0.169294  0.169294
2013-03-01 04:00:00  0.418399  0.418399  0.418399  0.418399  0.418399
...                       ...       ...       ...       ...       ...
2013-03-07 19:00:00  0.900000  0.900000  0.900000  0.900000  0.900000
2013-03-07 20:00:00  0.468219  0.468219  0.468219  0.468219  0.468219
2013-03-07 21:00:00  0.094563  0.094563  0.094563  0.094563  0.094563
2013-03-07 22:00:00  0.675806  0.675806  0.675806  0.675806  0.675806
2013-03-07 23:00:00  0.900000  0.900000  0.900000  0.900000  0.900000

[168 rows x 5 columns]
BEV DSM Profile:
 name                 BE0 0  BE0 1  BE0 2  BE0 3  BE0 4
2013-03-01 00:00:00    0.0    0.0   

In [3]:
(avail_profile * (1 + dd)).sum() / avail_profile.sum()

name
BE0 0    1.047994
BE0 1    1.043877
BE0 2    1.045348
BE0 3    1.057154
BE0 4    1.045957
dtype: float64

In [4]:
dsm_week = np.zeros((24 * 7,))


# assuming that at a certain time ("bev_dsm_restriction_time") EVs have to
# be charged to a minimum value (defined in bev_dsm_restriction_value)
dsm_week[(np.arange(0, 7, 1) * 24 + 18)] = 0.3

In [5]:
dsm_week

array([0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
       0. , 0. , 0. , 0. , 0. , 0.3, 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
       0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
       0. , 0. , 0. , 0.3, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
       0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
       0. , 0.3, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
       0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3,
       0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
       0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3, 0. , 0. ,
       0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
       0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.3, 0. , 0. , 0. , 0. ,
       0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
       0. , 0. , 0. , 0. , 0. , 0. , 0.3, 0. , 0. , 0. , 0. , 0. ])

In [6]:
np.arange(0, 7, 1) * 24

array([  0,  24,  48,  72,  96, 120, 144])

In [9]:
temp_air_total_file = "/Users/gabrieladams/Documents/Studium/master_thesis/PyPSA/pypsa-eur/resources/test-sector-overnight/temp_air_total_base_s_5.nc"

temperature = xr.open_dataarray(temp_air_total_file).to_pandas()

In [7]:
temperature_pd

name,BE0 0,BE0 1,BE0 2,BE0 3,BE0 4
time,,,,,
2013-03-01 00:00:00,1.058841,1.926620,1.566832,-0.010253,1.333019
2013-03-01 01:00:00,0.825140,1.785749,1.369204,-0.275674,1.124878
2013-03-01 02:00:00,0.733539,1.758221,1.229310,-0.232585,1.104091
2013-03-01 03:00:00,0.712404,1.619061,1.191554,-0.360440,0.962976
2013-03-01 04:00:00,0.634359,1.534520,1.180975,-0.514655,0.770232
...,...,...,...,...,...
2013-03-07 19:00:00,10.576603,10.372927,10.699649,9.224278,10.292632
2013-03-07 20:00:00,10.145163,10.064008,10.287299,8.811206,10.013093
2013-03-07 21:00:00,9.969361,9.919556,10.107186,8.597737,9.899897


In [10]:
temperature

name,BE0 0,BE0 1,BE0 2,BE0 3,BE0 4
time,,,,,
2013-03-01 00:00:00,1.058841,1.926620,1.566832,-0.010253,1.333019
2013-03-01 01:00:00,0.825140,1.785749,1.369204,-0.275674,1.124878
2013-03-01 02:00:00,0.733539,1.758221,1.229310,-0.232585,1.104091
2013-03-01 03:00:00,0.712404,1.619061,1.191554,-0.360440,0.962976
2013-03-01 04:00:00,0.634359,1.534520,1.180975,-0.514655,0.770232
...,...,...,...,...,...
2013-03-07 19:00:00,10.576603,10.372927,10.699649,9.224278,10.292632
2013-03-07 20:00:00,10.145163,10.064008,10.287299,8.811206,10.013093
2013-03-07 21:00:00,9.969361,9.919556,10.107186,8.597737,9.899897


In [ ]:
cutout_file = "/Users/gabrieladams/Documents/Studium/master_thesis/PyPSA/pypsa-eur/cutouts/be-03-2013-era5.nc"


# Öffne das Dataset und zeige die verfügbaren Variablen an
cutout_ds = xr.open_dataset(cutout_file)
print("Verfügbare Variablen:", list(cutout_ds.data_vars))


# Wähle die gewünschte Variable aus, z.B. 'temperature' oder passe den Namen an
cutout = cutout_ds['temperature'] # is 3-dim, therefore df not applicable .to_pandas()  # Ersetze ggf. den Index durch den gewünschten Variablennamen

# cutout.head()
cutout

Verfügbare Variablen: ['height', 'wnd100m', 'wnd_azimuth', 'roughness', 'influx_toa', 'influx_direct', 'influx_diffuse', 'albedo', 'solar_altitude', 'solar_azimuth', 'temperature', 'soil temperature', 'runoff']


<xarray.DataArray 'temperature' (time: 744, y: 13, x: 23)> Size: 2MB
[222456 values with dtype=float64]
Coordinates:
  * x        (x) float64 184B 1.5 1.75 2.0 2.25 2.5 ... 6.0 6.25 6.5 6.75 7.0
  * y        (y) float64 104B 49.0 49.25 49.5 49.75 ... 51.25 51.5 51.75 52.0
  * time     (time) datetime64[ns] 6kB 2013-03-01 ... 2013-03-31T23:00:00
    lon      (x) float64 184B ...
    lat      (y) float64 104B ...
Attributes:
    units:      K
    long_name:  2 metre temperature
    module:     era5
    feature:    temperature

In [33]:
pop_layou_total_file = "/Users/gabrieladams/Documents/Studium/master_thesis/PyPSA/pypsa-eur/resources/test-sector-overnight/pop_layout_total.nc"

pop_layou_total = xr.open_dataarray(pop_layou_total_file).to_pandas()
pop_layou_total

x,1.50,1.75,2.00,2.25,2.50,2.75,3.00,3.25,3.50,3.75,...,4.75,5.00,5.25,5.50,5.75,6.00,6.25,6.50,6.75,7.00
y,,,,,,,,,,,,,,,,,,,,,
49.00,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
49.25,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
49.50,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.195661,11.624192,20.817123,0.049091,0.000000,0.000000,0.0,0.0
49.75,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.617868,10.709660,22.490749,29.844514,54.096522,1.882881,0.000000,0.000000,0.0,0.0
50.00,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,10.638426,31.644242,24.463795,24.949988,18.762174,0.369473,0.000000,0.000000,0.0,0.0
50.25,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,25.438151,...,57.690801,43.388877,33.736806,30.143534,43.570217,36.520060,21.686215,0.598620,0.0,0.0
50.50,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,27.361679,74.496067,148.994987,...,139.383636,127.479091,88.699859,210.057502,213.482474,87.476841,33.620603,0.017207,0.0,0.0
50.75,0.0,0.0,0.0,0.0,0.896598,41.105837,64.766983,211.859439,188.105539,138.114953,...,193.007708,182.719454,164.355762,248.147353,171.376121,34.024310,2.361960,0.000000,0.0,0.0
51.00,0.0,0.0,0.0,0.0,14.333810,94.619650,157.259169,196.662560,246.736122,278.963928,...,238.347825,203.975074,220.626551,179.945928,76.353949,0.000000,0.000000,0.000000,0.0,0.0


In [34]:
pop_layou_rural_file = "/Users/gabrieladams/Documents/Studium/master_thesis/PyPSA/pypsa-eur/resources/test-sector-overnight/pop_layout_rural.nc"

pop_layou_rural = xr.open_dataarray(pop_layou_rural_file).to_pandas()
pop_layou_rural

x,1.50,1.75,2.00,2.25,2.50,2.75,3.00,3.25,3.50,3.75,...,4.75,5.00,5.25,5.50,5.75,6.00,6.25,6.50,6.75,7.00
y,,,,,,,,,,,,,,,,,,,,,
49.00,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.0,0.00000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
49.25,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.0,0.00000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
49.50,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.0,0.00000,...,0.000000,0.000000,0.001213,4.28035,6.654674,0.000027,0.000000,0.000000,0.0,0.0
49.75,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.0,0.00000,...,0.012092,4.512268,19.565306,0.00000,0.000000,0.039494,0.000000,0.000000,0.0,0.0
50.00,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.0,0.00000,...,3.493604,0.000000,0.000000,0.00000,16.260253,0.006337,0.000000,0.000000,0.0,0.0
50.25,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.0,3.28748,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,11.382616,0.008673,0.0,0.0
50.50,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,5.33463,0.0,0.00000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,23.825453,0.000007,0.0,0.0
50.75,0.0,0.0,0.0,0.0,0.009375,19.190295,0.0,0.00000,0.0,0.00000,...,0.000000,0.000000,0.000000,0.00000,0.000000,17.906805,0.135027,0.000000,0.0,0.0
51.00,0.0,0.0,0.0,0.0,2.112372,0.000000,0.0,0.00000,0.0,0.00000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0,0.0


In [35]:
pop_layou_urban_file = "/Users/gabrieladams/Documents/Studium/master_thesis/PyPSA/pypsa-eur/resources/test-sector-overnight/pop_layout_urban.nc"

pop_layou_urban = xr.open_dataarray(pop_layou_rural_file).to_pandas()
pop_layou_urban

x,1.50,1.75,2.00,2.25,2.50,2.75,3.00,3.25,3.50,3.75,...,4.75,5.00,5.25,5.50,5.75,6.00,6.25,6.50,6.75,7.00
y,,,,,,,,,,,,,,,,,,,,,
49.00,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.0,0.00000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
49.25,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.0,0.00000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
49.50,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.0,0.00000,...,0.000000,0.000000,0.001213,4.28035,6.654674,0.000027,0.000000,0.000000,0.0,0.0
49.75,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.0,0.00000,...,0.012092,4.512268,19.565306,0.00000,0.000000,0.039494,0.000000,0.000000,0.0,0.0
50.00,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.0,0.00000,...,3.493604,0.000000,0.000000,0.00000,16.260253,0.006337,0.000000,0.000000,0.0,0.0
50.25,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.0,3.28748,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,11.382616,0.008673,0.0,0.0
50.50,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,5.33463,0.0,0.00000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,23.825453,0.000007,0.0,0.0
50.75,0.0,0.0,0.0,0.0,0.009375,19.190295,0.0,0.00000,0.0,0.00000,...,0.000000,0.000000,0.000000,0.00000,0.000000,17.906805,0.135027,0.000000,0.0,0.0
51.00,0.0,0.0,0.0,0.0,2.112372,0.000000,0.0,0.00000,0.0,0.00000,...,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
